In [2]:
from deepfurniture import DeepFurnitureDataset
import json
from pathlib import Path
from deepfurniture import DeepFurnitureDataset
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import pickle

In [3]:
dataset = DeepFurnitureDataset("uncompressed_data")

### Bedrooms

In [ ]:
bedrooms_scenes = []

for scene in dataset:
    scene_categories = set()

    for instance in scene["instances"]:
        category = instance["category_name"].lower()
        scene_categories.add(category)

    # check if scene has required items and does NOT have sofa
    has_bed = "bed" in scene_categories
    has_cabinet_shelf = "cabinet#shelf" in scene_categories
    has_curtain = "curtain" in scene_categories
    has_sofa = "sofa" in scene_categories
    
    is_valid = has_bed and has_cabinet_shelf and has_curtain and not has_sofa

    if is_valid:
        bedrooms_scenes.append(scene["scene_id"])

In [ ]:
print(f"Total matching scenes: {len(bedrooms_scenes)}\n")
len(bedrooms_scenes)

In [6]:
with open("bedrooms_scenes.json", "w") as f:
    json.dump(bedrooms_scenes, f, indent=4)

Download

In [ ]:
# Load matching scene IDs
with open("bedrooms_scenes.json", "r") as f:
    bedrooms_scenes = json.load(f)

print(f"Total matching scenes: {len(bedrooms_scenes)}")

dataset = DeepFurnitureDataset("uncompressed_data")

output_dir = Path("bedroom_scenes")
output_dir.mkdir(exist_ok=True)

# Create index file
index_file = Path("bedroom_scenes_index.pkl")

if index_file.exists():
    print("Loading existing scene index...")
    with open(index_file, "rb") as f:
        bedroom_scenes_index = pickle.load(f)
    print(f"Loaded index with {len(bedroom_scenes_index)} scenes")
else:
    print("Creating scene index (this will take a few minutes)...")
    bedroom_scenes_index = {}
    
    for idx in tqdm(range(len(dataset)), desc="Indexing scenes"):
        scene = dataset[idx]
        bedroom_scenes_index[scene["scene_id"]] = idx
    
    with open(index_file, "wb") as f:
        pickle.dump(bedroom_scenes_index, f)
    print(f"Index created with {len(bedroom_scenes_index)} scenes")

# Filter to only matching scenes that exist in dataset
valid_scenes = [sid for sid in bedrooms_scenes if sid in bedroom_scenes_index]
print(f"Valid matching scenes: {len(valid_scenes)}/{len(bedrooms_scenes)}")

In [ ]:
# Resume from specific scene
# START_FROM = "L3D124S8ENDIZJ736QUPFX6UB2E3P3WE888"

# try:
#     start_index = valid_scenes.index(START_FROM)
#     print(f"Starting from scene: {START_FROM}")
#     print(f"Resuming from position {start_index} ({start_index + 1}/{len(valid_scenes)})")
#     print(f"Remaining scenes to process: {len(valid_scenes) - start_index}")
# except ValueError:
#     print(f"Scene {START_FROM} not found, starting from beginning")
#     start_index = 0

# remaining_scenes = valid_scenes[start_index:]
# print(f"Output directory: {output_dir.absolute()}")


In [ ]:
def process_scene_fast(scene_id):
    """Process a single scene using index for fast lookup"""
    try:

        scene_dir = output_dir / scene_id
        
        if (scene_dir / "annotation.json").exists():
            return ("skip", scene_id)
        
        scene_dir.mkdir(exist_ok=True)
        furniture_dir = scene_dir / "furnitures"
        furniture_dir.mkdir(exist_ok=True)
        
        # Get scene directly by index
        scene_idx = bedroom_scenes_index[scene_id]
        scene_data = dataset[scene_idx]
        
        scene_img_path = scene_dir / "scene_image.jpg"
        scene_data["image"].save(scene_img_path, quality=85, optimize=True)
        
        categories_in_scene = set()
        furniture_info = []
        
        for instance in scene_data["instances"]:
            category = instance["category_name"]
            furniture_id = str(instance["identity_id"])
            
            categories_in_scene.add(category)
            
            if furniture_id in scene_data["furniture_previews"]:
                furniture_img = scene_data["furniture_previews"][furniture_id]
                furniture_img_path = furniture_dir / f"{furniture_id}.jpg"
                furniture_img.save(furniture_img_path, quality=85, optimize=True)
                
                furniture_info.append({
                    "furniture_id": furniture_id,
                    "category": category,
                    "style_ids": instance["style_ids"],
                    "style_names": instance["style_names"],
                    "image_filename": f"furnitures/{furniture_id}.jpg"
                })
        
        annotation = {
            "scene_id": scene_id,
            "scene_image": "scene_image.jpg",
            "categories": sorted(list(categories_in_scene)),
            "furnitures": furniture_info,
            "total_furnitures": len(furniture_info)
        }
        
        annotation_path = scene_dir / "annotation.json"
        with open(annotation_path, "w") as f:
            json.dump(annotation, f, indent=2)
        
        return ("success", scene_id, len(furniture_info))
    
    except Exception as e:
        return ("error", scene_id, str(e))

max_workers = 6

success_count = 0
skip_count = 0
error_count = 0

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = {executor.submit(process_scene_fast, scene_id): scene_id 
               for scene_id in bedrooms_scenes}
    
    with tqdm(total=len(bedrooms_scenes), desc="Processing scenes") as pbar:
        for future in as_completed(futures):
            result = future.result()
            
            if result[0] == "success":
                success_count += 1
                pbar.set_postfix_str(f"{result[1][:30]}: {result[2]} furn")
            elif result[0] == "skip":
                skip_count += 1
                pbar.set_postfix_str(f"Skipped: {result[1][:30]}")
            else: 
                error_count += 1
                pbar.set_postfix_str(f"Error: {result[1][:30]}")
            
            pbar.update(1)

print(f"Processing complete")
print(f"Successfully processed: {success_count}")
print(f"Skipped (already done): {skip_count}")
print(f"Errors: {error_count}")
print(f"Total: {success_count + skip_count + error_count}")

### Living rooms (sofa)

In [3]:
dataset = DeepFurnitureDataset("uncompressed_data")

In [ ]:
living_room_scenes = []

for scene in dataset:
    scene_categories = set()

    for instance in scene["instances"]:
        category = instance["category_name"].lower()
        scene_categories.add(category)

    has_sofa = "sofa" in scene_categories
    has_cabinet_shelf = "cabinet#shelf" in scene_categories
    has_table = "table" in scene_categories
    has_bed = "bed" in scene_categories
    
    is_valid = has_sofa and has_cabinet_shelf and has_table and not has_bed

    if is_valid:
        living_room_scenes.append(scene["scene_id"])

In [ ]:
print(f"Total matching scenes: {len(living_room_scenes)}\n")
len(living_room_scenes)

In [6]:
with open("living_room_scenes.json", "w") as f:
    json.dump(living_room_scenes, f, indent=4)

In [ ]:
with open("living_room_scenes.json", "r") as f:
    living_room_scenes = json.load(f)

print(f"Total matching scenes: {len(living_room_scenes)}")

dataset = DeepFurnitureDataset("uncompressed_data")

output_dir = Path("living_room_scenes")
output_dir.mkdir(exist_ok=True)

index_file = Path("living_room_scenes.pkl")

if index_file.exists():
    print("Loading existing scene index...")
    with open(index_file, "rb") as f:
        living_room_scenes_index = pickle.load(f)
    print(f"Loaded index with {len(living_room_scenes_index)} scenes")
else:
    print("Creating scene index (this will take a few minutes)...")
    living_room_scenes_index = {}
    
    for idx in tqdm(range(len(dataset)), desc="Indexing scenes"):
        scene = dataset[idx]
        living_room_scenes_index[scene["scene_id"]] = idx
    
    with open(index_file, "wb") as f:
        pickle.dump(living_room_scenes_index, f)
    print(f"Index created with {len(living_room_scenes_index)} scenes")

# Filter to only matching scenes that exist in dataset
valid_scenes = [sid for sid in living_room_scenes if sid in living_room_scenes_index]
print(f"Valid matching scenes: {len(valid_scenes)}/{len(living_room_scenes)}")

In [ ]:
def process_scene_fast(scene_id):
    """Process a single scene using index for fast lookup"""
    try:
        scene_dir = output_dir / scene_id
        
        if (scene_dir / "annotation.json").exists():
            return ("skip", scene_id)
        
        scene_dir.mkdir(exist_ok=True)
        furniture_dir = scene_dir / "furnitures"
        furniture_dir.mkdir(exist_ok=True)
        
        # Get scene directly by index
        scene_idx = living_room_scenes_index[scene_id]
        scene_data = dataset[scene_idx]
        
        scene_img_path = scene_dir / "scene_image.jpg"
        scene_data["image"].save(scene_img_path, quality=85, optimize=True)
        
        categories_in_scene = set()
        furniture_info = []
        
        for instance in scene_data["instances"]:
            category = instance["category_name"]
            furniture_id = str(instance["identity_id"])
            
            categories_in_scene.add(category)
            
            if furniture_id in scene_data["furniture_previews"]:
                furniture_img = scene_data["furniture_previews"][furniture_id]
                furniture_img_path = furniture_dir / f"{furniture_id}.jpg"
                furniture_img.save(furniture_img_path, quality=85, optimize=True)
                
                furniture_info.append({
                    "furniture_id": furniture_id,
                    "category": category,
                    "style_ids": instance["style_ids"],
                    "style_names": instance["style_names"],
                    "image_filename": f"furnitures/{furniture_id}.jpg"
                })
        
        annotation = {
            "scene_id": scene_id,
            "scene_image": "scene_image.jpg",
            "categories": sorted(list(categories_in_scene)),
            "furnitures": furniture_info,
            "total_furnitures": len(furniture_info)
        }
        
        annotation_path = scene_dir / "annotation.json"
        with open(annotation_path, "w") as f:
            json.dump(annotation, f, indent=2)
        
        return ("success", scene_id, len(furniture_info))
    
    except Exception as e:
        return ("error", scene_id, str(e))

max_workers = 6 

success_count = 0
skip_count = 0
error_count = 0

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = {executor.submit(process_scene_fast, scene_id): scene_id 
               for scene_id in living_room_scenes}
    
    with tqdm(total=len(living_room_scenes), desc="Processing scenes") as pbar:
        for future in as_completed(futures):
            result = future.result()
            
            if result[0] == "success":
                success_count += 1
                pbar.set_postfix_str(f"{result[1][:30]}: {result[2]} furn")
            elif result[0] == "skip":
                skip_count += 1
                pbar.set_postfix_str(f"Skipped: {result[1][:30]}")
            else:  # error
                error_count += 1
                pbar.set_postfix_str(f"Error: {result[1][:30]}")
            
            pbar.update(1)

print(f"Processing complete")
print(f"Successfully processed: {success_count}")
print(f"Skipped (already done): {skip_count}")
print(f"Errors: {error_count}")
print(f"Total: {success_count + skip_count + error_count}")